# Fashion-MNIST — Custom Neural Network (PyTorch)
### Architecture from the diagram
- Dual-branch network with skip connection  
- Branch A: 16→8→8 with residual add  
- Branch B: 16→12→8  
- Concatenate(8+8=16) → Output(10 classes)

In [ ]:
# Install deps if needed (Colab has all of these)
# !pip install torch torchvision matplotlib pandas numpy -q

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ── 1. Dataset & DataLoaders ──────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_full   = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_dataset = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

val_size  = int(0.2 * len(train_full))
train_dataset, val_dataset = random_split(train_full, [len(train_full)-val_size, val_size])

BATCH = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH)

CLASS_NAMES = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
               "Sandal","Shirt","Sneaker","Bag","Ankle boot"]
print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")

In [ ]:
# ── 2. Model Definition (matches diagram) ──────────────────────
#
#  Input(28,28) → Flatten(784) → Shared Hidden(784→16)
#       ├───────────────────────────────┐
#   Branch A                        Branch B
#   Hidden(16→8)                    Hidden(16→12)
#   Hidden(8→8) ← skip-loop         Hidden(12→8)
#   Skip-Add(a1+a2)
#       └─── Concatenate(8+8=16) ───┘
#                     ↓
#              Output(16→10)

class FashionModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.flatten       = nn.Flatten()
        self.shared_hidden = nn.Sequential(nn.Linear(784, 16), nn.ReLU())

        # Branch A
        self.branch_a1 = nn.Sequential(nn.Linear(16, 8),  nn.ReLU())
        self.branch_a2 = nn.Sequential(nn.Linear(8,  8),  nn.ReLU())  # loop layer

        # Branch B
        self.branch_b1 = nn.Sequential(nn.Linear(16, 12), nn.ReLU())
        self.branch_b2 = nn.Sequential(nn.Linear(12, 8),  nn.ReLU())

        self.output_layer = nn.Linear(16, num_classes)   # 8+8=16

    def forward(self, x):
        x  = self.shared_hidden(self.flatten(x))        # (B,16)
        a1 = self.branch_a1(x)                          # (B,8)
        a2 = self.branch_a2(a1)                         # (B,8)
        skip_out = a1 + a2                              # (B,8) SKIP-ADD
        b2 = self.branch_b2(self.branch_b1(x))         # (B,8)
        return self.output_layer(torch.cat([skip_out, b2], dim=1))  # (B,10)

model = FashionModel().to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── 3. Loss, Optimiser, Scheduler ─────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
# ── 4. Helper Functions ────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer):
    """Train one epoch; return (avg_loss, accuracy)."""
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion):
    """Evaluate; return (avg_loss, accuracy)."""
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            total_loss += criterion(out, labels).item() * imgs.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

In [ ]:
# ── 5. Training Loop ───────────────────────────────────────────
EPOCHS  = 20
history = {"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}

print(f"{'Epoch':>6} {'Train Loss':>12} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9}")
print("="*55)
for epoch in range(1, EPOCHS+1):
    tl, ta = train_one_epoch(model, train_loader, criterion, optimizer)
    vl, va = evaluate(model, val_loader, criterion)
    scheduler.step()
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl);   history["val_acc"].append(va)
    print(f"{epoch:>6} {tl:>12.4f} {ta*100:>9.2f}% {vl:>10.4f} {va*100:>8.2f}%")
print("="*55)

In [ ]:
# ── 6. Loss & Accuracy Plots ───────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
e = range(1, EPOCHS+1)

ax1.plot(e, history["train_loss"], "o-", ms=3, label="Train")
ax1.plot(e, history["val_loss"],   "s-", ms=3, label="Val")
ax1.set_title("Loss vs Epochs"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(e, [a*100 for a in history["train_acc"]], "o-", ms=3, label="Train")
ax2.plot(e, [a*100 for a in history["val_acc"]],   "s-", ms=3, label="Val")
ax2.set_title("Accuracy vs Epochs"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout()
plt.savefig("training_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → training_plots.png")

In [ ]:
# ── 7. Save Model Weights with Pickle ─────────────────────────
weights_path = "fashion_mnist_weights.pkl"
with open(weights_path, "wb") as f:
    pickle.dump(model.state_dict(), f)
print(f"Weights saved → {weights_path}")

# Reload example:
# with open(weights_path, "rb") as f:
#     state_dict = pickle.load(f)
# model_new = FashionModel().to(device)
# model_new.load_state_dict(state_dict)

In [ ]:
# ── 8. Generate submission.csv ─────────────────────────────────
model.eval()
preds = []
with torch.no_grad():
    for imgs, _ in test_loader:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())

submission = pd.DataFrame({
    "ImageId":   range(1, len(preds)+1),
    "Label":     preds,
    "ClassName": [CLASS_NAMES[p] for p in preds]
})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv saved ({len(submission)} rows)")
display(submission.head(10))

In [ ]:
# ── 9. Final Test Accuracy ─────────────────────────────────────
tl, ta = evaluate(model, test_loader, criterion)
print(f"\n✅  Test Accuracy : {ta*100:.2f}%")
print(f"    Test Loss     : {tl:.4f}")